# ACE-Step: uji konteks dan penyambungan bass

Gunakan **VM Colab CUDA yang sudah punya checkout bridge/runtime/bobot dari notebook 10**. Jika perlu, salin sel-sel ini ke notebook pada VM yang masih aktif. Tool ZIP ini bukan installer model. Helper bridge sudah disertakan di notebook ini. **Upload piano asli saja** saat diminta; tidak perlu mengunduh atau mengunggah ZIP tools.

Semua flag inferensi awalnya **False**. Konteks **90 detik sudah dipilih pendengar**; 120 detik dilaporkan banyak noise. Jalankan sel persiapan lalu bagian chunk seluruh lagu. Probe durasi bersifat opsional dan boleh dilewati. Kualitas sambungan lagu penuh belum divalidasi.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, sys, tempfile, uuid, zipfile
from google.colab import files

for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}; jangan ubah environment model pinned"

# Generated from repository scripts by scripts/build_context_bridge.py.
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "ace_context_chunks.py": {
    "sha256": "1819d968044da24cf159720300ad9bd11f7d008ebd55036378f23447c1cc73cf",
    "gzip_base64": "H4sIAAAAAAAC/6U7247cxnLv+xWUzgPJ3R7ORRfbs0sBPorOiZHEFmwnCDAZEE2yZ4cRh6R52YsVAfmH/GG+JHXpJpsc7sqB9bAaNrurq6vrXsWXL19+//7D4pdWVU5cdkWq0kVSFq16aB31UKk6O6mibRxZpM5JyaarVSqcRp6qXC1knt0WKnWaNmuTY1bcBhcXP6uuUY2jcpW0dZZEhy7Po6aEd9Wj0xXJURa3Kg2cj7ksllVdxspJ1UF2eeu0pVOUTlYcVK2KRAUXPyqVNk7RnapH2DPJ6D9E8pDlSjiHw6lSt8Kp8q5x2qMChDNApbh13v/rP3zv1F3RAvbOoaydxUI9qKRrAejLly8vslNV1q0j69tK1o0yz0fZHPMsNo//CWib3yfZHi8OdXlyKvgFkxz94iO+0L+bY9dmef/UxXC+RDXNhRmiozgSzlT1s8x5cLg58B501qAB6srcbJSUda1y2eLBsyKNKiU/DZDP6U3wgBi5gkv5/tcPTui82qxWq4uff/rpV3hAxL0owq2jyA9q1ZT5nfL8AEiCV75b7y8uLuBu4Fhy8+athwf3txcO/EuzW9W0oaZXoCf49O4+a48MnBYEZaUKz61j1yeM2lrJE0PBf3g5cV4mn+DenaxVtZfLU5zKLU8EtGTqrVeb15f4xxex6/pb3j7oqhTI4dFy3rtWbVcXGr3gqB74F2DGJ7mvYYsI75WQE3cy75Q+Ew6EA97XrULKyvoxxOcAjxU13eGQPXhu0J4ql7fspwUMHCXHwx2CFC678WgLARcGNA03QuZ5eR8Vsgj/JvNG+VfufxRnkGpV5TJRjIhGPZePZdd6bdnKXDhaRMPvVgJYs77NivAt8IVMVRi8+Rbed6oJfywLYBZVqPr2kR70SQmlJvQ0EMEABK5mTLKD48ni0cuAf5pWgix6dyIuy9x34LqKsnVGb7ysaMUhL2Xr9xNQXoKsAU4Fonh3Pl30HV4y7+73HKAXrFc3oUboJny7Wpnx1Q3idcM44iD/ulnjb71gsbk0oyt/4K1aZo1y/g33+1DXZe25qFDMItjxf//7f2CnxpDwXbhumIiIp7fShPGR3LWCt6vG7Qk0oQLfCxDCHw50iWJ3E9IrOhMNPIffL2VXJ8o5dU3rgGI0GDqNAqRTs/tfnB/gBLcKaFGmKnfSrpZtVhYNoJ3kHaCvQFtXMk1RJTs/qjuYqh4SPDyqyVr9BjfQIilkFRDIk3zITt0ppGuDqwRkNJl8QlpLSBrWqK48Jgy98a+JOqGGAFcB0wyR8M3N+uvnNucxVwN4ObmSd2BIctCfgLUsgBqNUzJMR3ZpVmpyJIBSG65xP01qjQsoRdgHHryNoIMlKss9virG018iNF/DkZVMsjYD0djh8BVMQKAZcsNuJWifxXrPYAkNZGp6XaNV82iGvx9Oj4iF660NWePGk8AiwdBqf+1UZZ4lNMG8od8EBn4CqzkowXpvGDGbTOS0fVJO2z8gp+0gloNkrm7aG6LZkswInrnFMyMWBAd/vAgbMEMq9RrVevTmWUH8K7IRqDoHLXTTMzyjIcBPyIBF0dgjTHI+4BwZcDbyL1o3ffeTC1jzJdlbq5MEoMUti+ECKb6DS+wntMDKqg3N+BUzeL/scri7XQbvl0138sZj270/kC0vkfu8HpxmfsG7TxePlh6z8JQVw9LJzhoEM9ki8y816AEAcENevjtm2xnFVzoH8N6yGPyM2NAevb16qhFYApda3+bZKWuN3rHIhYgeM4FHzUvBgyMqCDyN/RYEjrAVeimPXm0mZygr0mThju+h5ddjpuOD3oSjKTfhMdvbpNCQtjZHo8AR7vql+KQeQ3Y4nHYr48ZrF+YwWiYfQxe8uDvgvTQylHN7mCiOkx2AWTP0S5owB2/UY8YkglytBaK6XIKVIsHZzZNxP4vyAHiMtccGHmQXT80PrCZW/lWwWm0uR+dajulNel2fEz0TXh8VStYRr3CnYN1Y5qhR0giFd6AFMm4gK3D3UmRif1BqZpQ38i+ms4mz9fCxKz71WpDEW3hSxD5evgL/WdXo8/2eVSQogqSFRHC4BlB4dSsAMHHgSsgFqnpiPDbTMar2gQbGfoZAynRBq5lM/QzZNEoHAWxEzBK/d1oGMtAJzNE+u+j6PbjbTLhoMiIC726lfoQ57jYWMKvqWvOSD6DHaAb8EaM7s/655AVEBiV3O4Pml5F//NnlAC5CWrpbUlAukSY61BIUsrtlQrn6cBHoosg4IVvjN16MsSCNMczSXqWL/tQwik/CNYIUGQ5xt+YX7EkEhG3o/y/a/8VAICKzPwpEZCtFU4fNgSMF8uplfi8fm2iThr/WYEXS9rFSoUuG79Vm8OCa+kVIFg2YDOFgGFMpCHtehBtj+4pqMI44xw/Agfe+7mPyEuBECGVLiLs+/ePvI5/FBCoAUp+vAqcfAi+vIRdQOODsw+3/MU+fgjjbw9dAGAYHNTxkxXmChnmKNUzreTQ4fUqz2tPxIBOTQuyo/KTDF55dZ4AWMA/Gg6EOBvV+NOEW1QcxYkQ8HTL8pccs7l7x5IDDqwD0EMSB2pxwTB0kZfWIpzT0mYIcbpW3P3v/Ihyh+dwN/qQnOrTS0SkL5/6IUTrfEyYZ0HBCnGbCtwQc1sKczNUnupd3rvbZ38uiLLIEwBIjoEeH6g0hoKOxPGUPCLUs8sfA+TujDx4RpgNoJ7ghVd+pVIMzp4G7Jpw404AQQPyTo+KMCJogBLUAdQRWDHM1lBIAjQhuNaVfGo4AhmxFUHeFt3M5u+IKd1GUTQuhBP68gz8KqUTj+CdzBYTq5rphQNY8wtZ+VmnBpATWbnB9spXwX5Wcomb9Nle8lmjp7wWcI/lEjMc0JsqFlirgiYPHjDLQu99GMoa7PqlWosyFFKCDQkgbj8Sgn+uTHuEo3h+5VmZtADbR0/cbMbe5wF1jth/7BM/4vwaqk5ZwZdojT450edYVU1iIfGu5YnTgMVZpeV/ESoLHJnZ7/2qCskrI6aF3YyhnvjsrV2UMv0dmwToV5VxCuoMdG1masWD/5lt/i8Y2V4VHU3x+e2Xe7i33nPQg6WYPlG3zW03/n+BaObFzebnxNT4oKqHOg1iwZ9IYAk8hGPdhrUkYNcBWJxlBVNyg1d8Ay8m2a8DRQ1lUqSt6VcG3OlYdQt89KV1b0M+ZfcQlhkWYZwUJdBuyPAYff/7wy4dfzyFYeszedqreggLoK84nz+J/tgmwHZzkR517rRWmesFyLXt2Cpz3VmSuswyNtkeNjs84hB+lHn4A3wfdILAf2i+6JoOWFV3WPnJSuWtIKf7WyRzHugLuJTtkKg00r1tJO6Nd6TZxxBX4c2RTcUDb1DN6eIcyh5iHF1lpPz2+JLgkLPN05+12rj0ErufEemLePAF5Cp8DpzXHGKAe3I+yTBWH6BFlf7M7FbWlxnfYl8IJO0/7IjRYPGfp/t6jZXSMMXYATjVw2dqOgEkYey64iclMghqN8Bi2YzbJpg6YXpMUW/qXjnemdBkdTr2yA7KcIb1NJi2IQx57QjILlE0trbBfhLMX8RzxBoP+R2kHmPIuLPsueJoj8Se0eID98Cl2Ix/9WeQ+0g4EkJEzoYCTZgcQ8oY9C8t/ORcWi2IzYjZ5qbWi5gmS4ceIZN4WOkEjmkfAawG7Y/OCkUOXqRChl8JyPmOXwbM5hWZBX/+IcQlq4+ve2PM+LIjscIrPX4ZMt/2aLQHacyOyYA6QivYkfXv9rZjbtFPaY9M7sSdm6Z/QDbPbcPgIwInKu1FQuR9HtGdr+xTQEEgCIG8ECUPR/WIOuP8V6PoMtzIjsME351MmUexwismLEZkn0gIcMT390ziP4GhlwADGHKFO/bU9G/i9xz0ccrMLzI8sgSmwGtksDe5LNL/OKWvIxxsrVNzHhLpdwZLTeHVZsnhFlm615UnrSUsH95NtPfq8JWRR4ELiISnafCjk9RIZDCAC9MtcU9mTV1zZw0GrrodgsJKQfPLwj+CBf/7p/T9FH/79v6ynH/86cDQQ7xaESQe0ZmszyprgbPLImNgARiqDlO/oLYWyjedzVuvzl1FNkuiPOSeWQZ2T2I/9eszDUbhiuAszPXswcaQWDf4HXh195mkvadrL/Xb1Kv3ijhNxtE6H3X2kPYQ/VjQCWxN2+kTbM58ukeDlpoNnoyfuYOF+6rBMYKM15eXnNpTM53bWGuICMgu99aMDu+d7jKwD72Rbh7P5FcTbIIrUWuAY91BDx2SoOORdc2RCXWv/Uo2g/MX5WSUl1sEk5WYaxFW2WHJtHXlo8UVBPmxddxUmwg3B2P0MxinZErNn6OH18abNOwZuVugChqZbcJuXMQgMv44uR/pkiIjIKOpJM5Zw7sa0wsqYs4Gl0bO2xEK/t+VhbO9C295tZ4P20aVp9Oxbux6oot9ex7Dhpynn9tNMTet8O63VWfFXj+0RXH/yJVEdgijDdR5kljuxOmAVLoFtqO0DrhD3fTQ3cB7odKdY1eHq7AXndvQ9gcSaO/rMK1hW/V5lbHn4Klyfy11PheeBPXVocw2s+nW2qwcqyHY9bd/n8y1Dgvv/Zc6F7pmZ0UDnUtxj+JQgjzQQq02zJjgPbXz/2m7WsBW3ME9jf1QPahuaxp7d3DHKM2xWl0UFknG7XnmYvOBRni7WarGmjIO2xadG5xlmAE0TFvTcYRaXlujMM4zS9Lev/QFuhR1TI8insig5o8LAJPBauPZ32+23+0A2CMqzQBn3FRMhuNS/efuaHFVAmJ/hJK+3fcb/+Lu7pYYQF5u6yjRLINp2t6vgS7/7IsS/vLvOtyVh33ZEUIX+k2JCvctzV4ATdyxTeDqA57nrscGyqAaxRArosre3FsMUGJa6PGbGtGufl/fimN0edYnv9Wq1Wm7erLiGY429euMTPJnAhjrRg21RIgqHHil4uwOAWwSIxhnHrsBjurejR4RCbyg2hBWr/U24+sPki7EdipkC1hKgPZ4ZFO11Lm9DtF+Ie8U5XzLdOIkq+LBg/y5EEJfBd29MN0JZHLIU++DCntmSPKvoNPJ2vyQcxUqs/UltB1El6sA0VrgG0rswePOG3Z2ZswzzTM2FSjf3CugGSs+urEtUVXiteVY0FbYqrQQ8VdlywxX4c9a3ccSjlHAvCAUYXCAyewQAOnAyalrQlDxF93Cl5T0wy6EVNSIlctIzoub/EoX2W9xnae8qa3VJLxb0YrncXGNpkN5c0RAfKUS4Oxpe5HNKEYtw82+Y0+OQkNIg6idB1E+DQG7gMhSEXoSb2BBDxmejM0X+v2HnElh32KQBjnwswQmIS/DHdIbU1PHvpbiPw9HlMs2u4xxJIy/v5VV8eR9zNksKBUdDPegL/C/2r2t14FYhUn+ekpdKXqn4UsX+cmP4t64HzmWNNqPGLmEFrY/ZRUchoR39d6C/vhmYlfkHzVgIup3wQGT9JeDisw90sJasAl0n+Qjj4N391mWKqmlam6VOC6qnybgNAIQRovPsd6XpheFbnnO+BF2puMvytNO9UsC6bejRgbHAzvV1wsy/8vDZWy9wH193QNXo6KCOGRp4VpPQT9erBR1JfHaZYa2KKz2zUXaRTyMgQJTGB3gH1FDSKrS4xIWTCbFP9Wa2s1gjxgfh0n6Dh89r3C0dxoJY1QqUBOYGcnUHLhknumgqAV+ipoNb07YTt2pad8u3j799o1G4UziSdS0fG4pVBZm8RsurLv+Pg65r4k5tC/jVqKi8t3pItEXkpRDK93WCZk5iPoAP86hjPdODRNmconXtSigqst9VDTrLm0vEgThOFN6rzdCfxMpJF/0Kh1oXCL3x0Y38kwdgpP0P+mtaS0wr1uxeTEvWz2UvGEvafVmgPaAy9iRjQepSxMYLtdoa9sIeI3yHLAtRcie38V4XjuTsWbbx4mkFiXZg3B1iN39ZN2+dd7AXjLFupzK/99dFecLknFbfo/MMPSVKgt427Y+gpNbBG52fmzQ+7Jev/ElTk9UT5O30bvshG3zeDbEnFMNnW4/sf6RUmAoa/IIRFvrxih/7DqTNyipuYnDcSqCjRD8IdLRzlPmBctemIdO033Zt34VHfbKwQkH87qj01oqR2zo7jcR1SiKij7EUEwrtEpYa6vbqh0mX9hb8nbmrEYtc4b6z9EEdnlyxleuB+Dds8kfShSBGbWScDDFozLfWIVdyuow6GOAKi5ZI4qRZk8g6tQSHPEVkIM/2aVgTMGOa33theTqJdmvmaOPbDWEP24fdq/2OVfB+JLDavGDM35C7aXHAD2B6sT9AaAdC3z5YjESX7twG7rWjJkHKk2TYklfSvELdEzsEU2EfM+NWasnng07eLuzLMLpAzo2e6ZR4O2FzotWw2X4XL+YYZnbZ7Mx5NUb0tG4WsxGjE5r+s5lD6FL5K/9rZ6Ts1NePMj+sS+6vBnB5Jeoq5PCTMfaFDkZxI0skqelnvVmtLsnVg3h549XVDuOL/TLXP8jJMQ8kZ2bKxHXT3njbmCo8pUHDzGgooxxC/czODtt5PAsb21DyELotPBA/0RdHXhIdLMQjk0/Ej3XFBzZuDB+U/j4B6ySLDjCs1V2m7kOPjrF7wnnaOzfO4hUFkXFjpj7tQO39d2+fauwb/UOxpy+xbGcS6Y3b0BvwmDFEnofGJf6PeO4l4aCrlVjEf0S3R9fhA+cDVuIXFfZgDU0ptx1G6q0C9+goG2UV8QNLt5F1Np2PdPQhJrdiARAKDP0Bb919pm0RVodCtq0CAuElLuO8PfxgdloHtuPMyy9xHTjNtLu7pf+Ei7lWbHhQMq3L8sSlpy1NxeugZ8oPuFv8+2R3peMi6dyt+wt/jZVm8hYborKkoV6ta/xmTXZtCUKSJU6Sy+xEnykcyxouJ0m6WiaPAhvd45oI11BPVvJIJbBx90PgThxlbMX6M+WfuRqJN19XmamwstKx3a2vlkUA4fOaw0yNZLYGoc0twJjvd3i+lDLn4OuanF2UGMrxwxENz1qdZaPsJqyx85r+KDboa9Z0ZSp1TSCiuyj7pPHM9z56iZ7KHzA2132zn5O1JsktgTJHEEnexbUuSOiSwdOx1XlPpy4OHvijtb6v5ryKziDIaxRNF3Mr7cf3/xJtXrtPADllD/QVIq8P3lx6wTeXFmV1T8OVbt96BjRyHNZF6QOcMXKu6Pex2e9PdjHqIxzcz7jtFzqB3zcp5ll8ql6Bn6VwSUxj6+82n+bXwlR3poWRzoT+IsYtdHE7rbmsY3C8r11ofLubpgT2kxBjegsHAhp9psUjObGuVfsnOj32ikFOW/f47ZV+u3/uvnSZybTZ6a66Xi6EesD3OmwOrT2sIrvpknuaJf0n28LY/nBfGOMyyrPwkFaw+H2PiYqr0HwJHHxf33b4xfNHfKq9FLRGndHHImEUpWUSRf41HD4E+5WmEbIbTWxwZhuC130CZyHF7X/rMrD61s1T4xOsoJW8zHO5AYeaomhc6v09d7FgQXEFURrrXxOo86tK7G95asnQCzldZj6lNAzGIMhoC/2Bdvjd6ok9Ob57du3bZ/ZGl+7ZxcGbb5/YGRs9reMOMoYNkaw3qMsZRBUVv9DcONIZ00vBpXDLZ3vRXT1PWmPFAMILw/ngkpyBwuKeDcocFL/NnttaV+JAbqmHF0FjlgICMuWe7cx5AzjzzMZ9Gw1vjl8EgqeSJRh4v12J71YCPP79lNHO4XB8MXdXOrcJkxuQEiIprmy8PkGHT4GWlJC7bl37i70GwZjvIWiyLoPSb+BvoSHwtymacXhwnGngMTtZqJf2XyCrfAYhJp7VD/KMw0Vrhy6aoY1jaP8geNFnmmnozy7L9R/p+OBAaahZP1mrpolnpWoatSvVmsDTejOhR0Xn55uFxLMb6wI13xrltfvi8+j8fW2ZRvXD01eCsjtOk1rLtvpEVt/UcJiZ+6GP9iLBq0T09GXSREOuwSGfzOLuFOuTfw3YfPNPNV9AOYpQOKMIjhNFaH6iCL+SQjN08X/qghc6h0MAAA=="
  },
  "electric_full_song.py": {
    "sha256": "69b9a31dacb3dd8720466cdffcfa7e88c1df12011bfb335755ac3cf3e038494c",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoRnRpGlWD2xpLHsTKeqBgMCS2ojEEBxoUy7muk79A37JP0uu8DiQkpx03oyMbH3/e63teM4NypZxfIoC4pCHE//+z//64fptBBnsQzLXIXiZ2gfC5WEcRXBSFEVUgSFCEScJlJUWZwGkYzEaRoHC3Ev40zm3sHBp0IWoryXQn5WRYnzMpUkMC6vklKtYYkkEjJZKVjj5PpiLB5VeZ9WpVBlISIVrJIUpoXi9NeL74qD4+nkNZwpDDJPXMGMIq3yUIowTUr5uRwLPMk6jWQMQ+L4R5GkMAT2DO+r5AE3T3MBy5XhPXx4B47jHKh1lualCPJVFuSFNN9hmm3N799hDfN7HZT35ndaHCzzdC0yaIvVQujma2tIUS2yPA1lUdQt2/on3p9XKLcZgEk338DfsbwM1rLIgrA+UVWpyPx+DDZ1+xeVLVUsDw4+nHw8E3Px+tV0Oj24/nB2c/YRPr8eCPjjFOG9XAf+RuaFShNnJo7HwlER/HCkxvDRAjDsQDNBEHtg96KU2dHm+A12SmfMi0Vyo0KJI8IqCnBKlstQ6ZWdJZBC+foVtqukKPNqLZMSO8wGhZRRAQ230zu94gouFySh9AtAHK78V29K05cyl9QO58Apb7A5XeIWkV+mfphV0Poxr6ReCWij1Oe4KIAWS6C1JWAbLl5uY6BDQ894GAH7lkEugjBM11mQKDypJ05ElG/HIgnKKg9ikcVV+ADLlEheDm1j/XGQYkUYyyCPt0BHQK0VbxuUZRA+ANMU94ioJC2lyGH/AHkC6T4ATlrEyDzALOI+yNdposLCG9jkRBREFmJdFQpgRMePkWtUAjyYpPlaNx7pO+VyBRwH6y7TOE4fkfqRDzMVJOl3xcAWvD/cG+gSR+MJo20SrPFIgPqng4ODSC6BK4swBULy8zQtXaSLsQD8A6ZyGc0vAUijGePCcc4VXlMschWtgE/vZfiAzG2YHEGySNMHsYrTRRAXyJ9ASRUQ6UYCNIEakgTPYpgaVoCzOHx43BooHPmNjjHycgko30h3RP3rIH8AtMMQ1wERsVSrYgIAkmVB1OwhXyM5lmkaFxOA1dqXmyCegGjysi0RapirrIQ+WVaZr0UW9vEGCA2YoCIXRYC+Nf7JYXwOeIljl7rEBIhpDQdUhY/M6o4AKzm1If70QXlRtWygKVSBMBII1GZ1WlHfux468uRnoOAIxHLudiGh18WlBs9LZw4UgPMD3/Esz9PcXTofrq4+im0AGIhUpmJ1LxbVQ5A0mGTMzsRXXPHJGXVBgM3UBpiMgMdLifgAeOqTIRiAQCOgUwBEpMLSQ4n4ILeFq2mLcAXCGVhzZJ0ZZxIkYJ7LS4zNUhNAHakGx266AQwXv8hE5gFqoqP3SFSFFmq9P4c80UPSdB1UI+sg81Hg+YcTvfpo3BkGEtPX2sjnMf6hM+pAWrWpxgKNF0Tc2uCPwZTLINoC4G7pxvbVm8nta8AmhvScjUyiNJ+cnJ4d3aA8P/beTDxo3EwWKplkW2DGxLGIs70SyoG9SxExZKlKgFO6KmNCnOsVwVKWMinSvLA3umPSuE9BnRBdgJyUkcu3hUs2l6t5I5aJqyeMxN/EscV0fQJ2TiIQPnIBGM+CLtF64qJQAincf3dx+RG4OVkBaRdBHNzD/8vKwFjN+tLS/vO9QFnh/Q4QcIFGXFB5Y3OpUY1AYgfdCooP72IuDlQoice1hCWY+YwXF6WsJiBuAThh217EOtDrOjdafPE8EBDSwW3TwiPJM58LBxQzb+/YpHBgCQw9ucbZfoDrLy2wFzKu1iIAJGT4v9/eIzQ98c8A4gTFSAFjSLbW5iBYAWKRLmDfvSB3IlUrj2JyPAWmA7PTN8rdX1Zx7CPPeirbJouxiIO4EqCXwdIUKxIBhfKcFm74ohoFpLvvbeCD4eYhQdzOAHnAiYBl1+Cho0FQLtjdWo04QO+8+jJH686WwmREoFnnpRlQOM6mXiCsfOGMyNIGayGd2fLcpSZvJUtaEMQa4Gcs6tYkvAf9CehttRbBOntUESiP0Uj8AyhHNBzH4hX8N6gSfgPQGfSefc4AwqCY0LKQKRibD+++iOvT98c/9GV/cw5935G+PrA1GjogA79IH9yHlcuyEkwPsMNlidADQ2oOtl5UobBOk/kxfKwClcyn3l/HoDE/+3UX+CmNxQEnjFWoSnRLVkdaFoP3wTZ0AUdPE7AsFluwhkC7Pqaoen8EsbIKwq2IwfQBxwO23wKpAgmUtbVBhjoQASKZ8G3s9Vz+e6VySY00coMAI7ODrtFcgm/QPn3Na26QbF2wocFiRkvY3YDmgxVHKAeRD9tdLpwOdBta2qN6CDonwKhg7iqghA0bGRvUEnyktmAnXwjOJ34SU7PCFD7MycRP89ZJ8RtAbQ/F62DzcU8q2EQDPqDgM/GGf7MR+9Pc3uMnRCZpHFoaVft0fHynaavxpjyQF+6tDXnXWS7XmVwh7zlH6DWCl4py+WiD/5d4FPrkRuUwizLhjYZtAOeI/BUahwenpUvdUiMQG4MlWhrOJgV5J+df8fBPzq5Fg1wvgYzH00M8FHlMR+GMPapw7RfHP8RSj2XOGN2NWY3N0eVhqACRVmD7z41U0SNB8OD6hryCReHqkUc18EmD6oF7MLh03sowRe++poW1KoDawvsZkT/q/Eh8Nb1PQOhAIV95v6e2mOVGLQnAfEXLruZ/0M5ZNcj/ll/BTncOPkiC7GD4sLgPXr35YSwec6A0n5z2P8a3ACWLyTTv7uJB3f0CPmTSabHbc9wSdPmlUck1BizXgE0063CNwNl1/GZE6wb7dG4tNfpXbOhJdx9PUSx0JMezQqKFbBJe6LE+3qcxCpAY6SNvYGNiVGBBgJiPCg2jNFcrlQSxfx8U6CYxWRhGZzAmyxR6kEQ8DCQUriVZiL18JsS+iIFRC8kypiNXwMl/9MFHyZXc6VM4S/TTy7m5KE1MUXY42hm1ZBKwOuou5nTN6ptAgZ0To9tLGHPxKrd6Wefu1qlXvhsBnxP52MZcG231coS3ZvGfEH/Q4trEVhNaM+5viFVv+mYvYm/IKOthS6RLNhB+JCSjwUY4hiOCdoad4LAgxPEka5VU4AK8gAeao30v8GCWNtt7RA4jwmoUqZFId2CH4rka6SY/hzLPSkNlRB/e+iFSaKmhc1oQpsYc6fTTh/l5AIY1D8fwEBruPA1tQkOmHhh9bGCE6XqNlwDb8k9UbH0ldrcPjLP6GN/DOQa03V3rrDyItJ9RfmCf0UG+ScsRnPQeHW2vt+zrP/KlG/XHS9hEDyR0SCoO5RCP1qbM4XOar1Z8RKoYfCuROHeInr2kCfqX92Z129HCSK0vOYemwh6t1nSqj8IupOE1MEV57+/N3sdgH1gGArhnFNSfdyYemvE0ndHCYQ0dhGoImtsbch7waBg1lktjmGDcGWjtgcMf9XA8pYzsQBh+e+BBAgcG68J1X5Efww4NRoovry7PkM4QFRjizYGaEEaNonu8R6GziNPwQczm9Yk8jEFoiiICnpndyLzgnjx4dGlmK8qmYblzgvNvUwcA6xqYH4p/1OdBmCukCEbFVx21/60VtNeYhq+vjbSftfGGoUvkdWinv592mKILGZQUiAdd40TpY9JqCIHAIv01PP8xyPGsNMY5QRY5Iu/KvghlYVQCxhpKUY4tTx7kdkLLi7UsgygoA8+5e2K6qc03tyEue0ETt7XbRrunahMzMtO+DudCdoKyvnlziIKAf4I5Bovqn8bWEJ/Njnpgywphwn4aWhru0kzVk/aBYfSk+V2DkcimNoHqlVpGEZ4TObG3kebPHq71LXvjO2yKgYVY+hiGgDHMhA4JDp/JH1rpc2AHC/y2lDTZKH8Ppcsk8jUvNfvohqG72IzR8JOM7D0a89jcHV066ADttodMVYJGgSY0g5KW92Matf8jP8sQbBufwxAUMDJOUBPS+CAx14rWCJjvOv6iU6ZgfcB3vOW8qdD68ghkUoDpEkqFTc6vX7+a6AgIJm5yFVlZFDvStc+9Yru4ykA+y2A97GbpKB6YwRuVpwlm02pvq99F8+yh86FR9smsnInrvDv333362T89OX1HMh6+/+XsY/P9/vrXn09O/352+VZ/nV5dnl/88vbig2MZrGnh6c1ucWkM7Vmbc9tBe6BXZRiTdvUBrs7Pf724PJs7x7DPxw8nlzfnVx/en324sXu0gG/B0N0TxsUjB5hhaMUePSAAmZeo5uz44tB8rVJMqjnNw3vbJqIGD7MZGNmt7ebnwrunn96eiCpp7GzMXlGoV2OOpP3p9afJ++sbsQzieBGED07rLDpF4IHZEsVgv2BoExhMf9ojcS5crbDbYB+cwfR/oI1rTOWec9b5HS/jWkt6J6FEuOgu64rIgj6lktfA3j5LFqRkv8lpuIWMl2NxGOQrkDaHhw+P+Kuf0uF2DHe6JkFO4VXOkc8GwWmXXKBtIuuoRCEQ1E7bIceTeBEWCwCVMgZ1rr01LMvx2M7ttUnK3wl7o7fq4wQdy8lvJ2fzJle/jKvi3jKrO8HcosoosfgsuLqAagMbwONvAunTLTRoGVo6wmP2619vN0YBGm3stwMBmoxglCEonP8z/7YoyJS66B63u0CXcHz0on3/RQRSg89M2gmm1tHZrqWMKaCev3wEXWs4RdLYAt5/CvzDHbd1oQRKPK4Wsdp6szRWOgfbf4vumWEj/tLZ+x4uYEAHBzTyL+IUBAnaxFpduuANbFIF/okp2/nOLhZiG5RqhY7YTOC4vqcVDkoOz85BwL69lAS7Rdp2bDk6ppHV50rnk9O2e980d4fpmMFQlAB0tMyVVoX6nFkcJK7ZUiuFcV1Kczu960412gnD6r+nC2MoYWDdsJpf2zxzU8GDzEX8PDe0wFVAOyx/INFWDY4OfbTLeOaYqIFV5k5B4sdH8eP0THULggA46yLajmpadBAMCA3g0ymTco1Wne/Xqprs5hqQjeAi+dPcdjdABNeQgAxM1BKgWG+4p8ZEizgN7YMOEDX0gHrmDTDGNfXNG/QPIbVerQt9LKFCjq5ykLcUP8jnli/BPRofZb5tRIUmPcNtzONdNGAkIiuxLFCe0U9EIGpmVG3WWj3CBMO7rIq5swQbArhozDPmS+cr6QT6QkGJppfvP4EHRU2t4hLUpWwSojSKrbN/I2EZcxxtDJ+CPb2cZJhWGCmnJOSxnYp4UXLh/yN3VeehAhCh6zk6RuyTzb/S4Z/GnQTV/jCdte4fTkqpZTcdBaYQHWJflOuKqTMC/49KD9AkyjdSF5AGYVkrIF6dV3QM+kAOPAQr6bYcqJejCN2LNtLJ0XDrCkk/xoq6hLz9cVM46cM5N0o+euvsNaLQddbqM5UgmJH43Rpk6eX/A8KomXxvdrOF2Vgt4GAxwJRWWnDr8T+9euiuuRvnIKUAriib+RcVjBRu23fThQgasnWdBqikoopL74vKHIRgDVtUdD2At/HQgvZoz23t/ZZSRqgL+jvaiBrEnYU88rp1Ba73ryo7xxKZGlR8X+czBas4+IjZrXr8xbX/9uz8V2C8t1zhASav2si2zbakmsgY5NiGHd9DDWc0ORGUYDPriRxqbPY308b1L8MpbQTN7AiFa2vrukbSqKBJK8RhmTmTnnxt3cIeWTAchhaZTqf+CeHcxyLh6USThRmqldZE/61jYKaqhhRKjVm3G0f5u5SZCESxBo0hMNqJupe8WAaHCJYYT7+6ej/BpWD9phx9HRRUQI7VIWFQgWY38ZNeIHwHjbVqqtCXJwuwcJ+jJhz7pxCRiZwM4b5bOdgiYatgbyepManTaI0NjVWpg1rdDP/hriT/2Fj52jLWxDmwCGYrgswFobXACrOZroi1il+Br4cmmmzqCyNfp1fv3198vBnvrTD4htAXayYyWn2tAqwp4IWsVjJq7VDH4rAQOkJL0KYqHYu2kMUaFsnoMi3PQVdGpqBDpzxx2hLbwcri6cbM4nAErWQVHBoTt/VmgLnQKl42E61yZ0yBWfl2exBmWnwMSOh6NLa395kJrTcpJnASqeUSq735QQUYDAUdEpw3E+hvFJHWsREHEv/s2vBZt+baBOrMtqM+juzrUdG1STW23s7oOnr97IWDv2m+/bactBZ6lGcikxzzGSznYS/zEISSHJb7xroALu3M6JGMF4FUxR8u462nfh3jmPmYwVU4TzPULflgmGnqeFS9RMOzHlZ/Vw5O4TOgonF3da4HK0fxXviECCsEVVKlFQXGmqdElLSKZAZGNLJy84wIqcG8IBK/nZxRwUKpKBYK6oPcEU+8C/LoEVAADkpJxG/ebcAGBVkv4RafIoEmSiTml7k0q5SEaM/Zk2QYUM+MygFPzkoLDZdatQIaY9EpWLRLOnGHrguHwYmEJpJZMG+nPL71BhbnaFUwG4hvLp3rX08uZ/Tm66vZ+Pa7drrou7unwuCUQtO/XH8S9aui56OefKyD7s5Xl2fi+uTmZmbv3Ow5814vn3TB7viZw+noOVNo82KtprEdZ7SKRdA671eM6xJkUok+CRvftzQjORRHnHYii/8IZzlNXoEHAM5a9v+oiQwSqVBojAWfzJc+uWMY52zoplZkTQEH6MH5zkRP361oyO/W4QOb8AfFLgd2N5VPWnv2QmuWInKHQnY9W3bUUlI2kVojKfKvJSlH/o0JFDlYTYUvF5rhtw5o7wLLtGDk8YvrrlMSWejTUxW2FdZnQlLJJg057ticE3bCgGJv79vpnfWchp7JDMcwe3a5Y2EGDXSKoBqbpH6NA7/HVAHTApmeYUrKSB2Tkb8jdYIjYKXbThb6DkfUrNXtfA6gdkJEH4h3ZgcHbQa1VAxJ7VhYAOUwNpUggD4ihxeXgFPIsqSCiLtbhwehfhvq5alWpF0aFKNCbl4x9kLz+v0k5qLHQw8ndz2bHK7gsPPqNSw7KXfcsgT/h4L4qDdjuUrbrzuDuPUKs7cNGNTgZ/n8TFF9MVuSNYKrg6RD98pqAhcLGLr0G0hwz/D6ZjTeoRhYBR9+VMFKmq4nmyCx8J6RRST4ILdEfVQzT+YitIz1JxiMBlGewvdk7ujZRxMnrNtbDxJoP+DhvNCG64DdynbWxBT8tJiIyUfLHHpOS2fGh0X4bsrqXWCZtl+oL5qpjrsDEEY5COZ0TbwNo3TtGkHquct9SmrCteKtuNCEdhY10VvKi733nZJ42MsflsFk4aEaxB+ELfpBL+N4Nl2ShmHCBAQugg8+DXgoHwC3xudQoEEx74gpSos10SNurdaS8rqHhXxrmOajnkTrPsMYmkS8xnNaqVDHlFjTjborISmHWpDuXSEcuAjbPHgRAiPDEDYZvfxZUKOasOiEddJE5x5EO+NgANuuz24L9W4h013vAUu3QqqOU1mFWaO9muLlS7ZKdjuLtgqq7p4F2AUpGXyeteJXFejd2drcyh+ObBXRpCfqd621dkEnFQ09FNjN8Yciqc25W+VZdzu22r/YQG8rPrtrs+Y913M3xBhsb7NOjP35Kw27MI2NNq5F1byvsDGFR1Tsayqe6793RZstHaxzdbaV0+t1doatMxk88LNj4EW07cDIWmzBUWwtuHPUSxbWaZb967YH7V6WyJAZYd7jpT79gMGPmNw1oR3b37Un0xsWAFpPBGFmmYN0N21vxoa8mDbmz5PMt3qvnUTUwW6v8oVZ1GHa/V9kUHWAe5e933WAbImtp7ZiWG33jD/JO7MUvJk3qMa/FdIY7dgV+LekCgG5BjCWh4KMFX8RH4AEFL/0qDOLNW9OWPKzRSPNZO8PZqFfdB397hvOYiLyVNaC/pgpcfFO8hXZ2tfU40aSI5AYN/f9KA3Bv7dm4j8l4Ad6itv490gh82vKKcCWATD6fFeMQP+jC8Xt8d3elet/aqFee+9wjilY5zBhUcvn37UT1e7qyWTONLeY7p1pvbiyJ+O/UzR3rtaq7Dw3M1GydfBZras1vhYxFuyOHXSoimppCCmgkFPM48Kl9s+sozADE/URayK4+XSNHuFNqxRGr0t/4com54q+DdYk8gZ2TUar6pjGIHnYiRNqBFR1/yUK2fIJzA5F/YKKHxHpE0m2dgyJYL9BttNVzHCPOmVUn2hsL98cyuSOuLMds+TGpoDbZJOoWX+MOhE954Yk6qzmyFtjGaPv+x+C6xKw25wAWVZhDSDLWHIefC6L9NGXJk4++B+dGmjQVksAAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="ace_context_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
# Explicitly load this notebook's helper, even if an earlier notebook imported one.
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
embedded_single = importlib.util.module_from_spec(spec)
spec.loader.exec_module(embedded_single)
sys.modules["electric_full_song"] = embedded_single
spec = importlib.util.spec_from_file_location("ace_context_embedded", TOOLS / "ace_context_chunks.py")
context = importlib.util.module_from_spec(spec)
spec.loader.exec_module(context)
ROOT = context.single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None:
    raise RuntimeError("Checkout bridge belum ada di VM ini. Setup notebook 10 dahulu; notebook ini belum menyediakan model/GPU")

print("Upload satu file piano asli saja; tidak perlu ZIP tools")
uploaded = files.upload()
audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
assert len(audios) == 1, "Pilih satu file piano asli"
SOURCE = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
SOURCE.write_bytes(uploaded[audios[0]])
OUT = ROOT / "runs" / ("ace_context_" + uuid.uuid4().hex[:10])
plan = context.prepare(SOURCE, OUT, context=90, margin=6, fade=.58)
PLAN_PATH = OUT / "plan.json"
print("ROOT:", ROOT, "| PLAN:", PLAN_PATH)
for chunk in plan["chunks"]:
    print(chunk["index"], "input model:", chunk["model_duration"], "detik", "core:", round((chunk["core_end"]-chunk["core_start"])/32000, 3))
print("Tanpa SongMap terverifikasi: batas ini bukan downbeat atau bar yang diketahui")


## Satu probe durasi

Opsional: `90` sudah dipilih, sehingga tidak perlu diulang. `60` tersedia sebagai kontrol; `120` hanya untuk eksperimen eksplisit meskipun sudah dilaporkan berisik. Caption, seed, FP32, CFG dan langkah tetap sama. Dengarkan bagian waktu yang sama, jangan hanya melihat status generated. Jangan menjalankan beberapa probe sekaligus di T4 yang sama.


In [ ]:
from IPython.display import Audio, display

DURATION = 90  # 60 / 90 / 120
EXECUTE_PROBE = False  # True hanya pada runtime CUDA/bobot yang siap
PROBE_OUT = OUT / (f"probe_{DURATION}_" + uuid.uuid4().hex[:8])
probe_report = context.single.generate(ROOT, context.generation_source(OUT, plan), PROBE_OUT,
    start=0, duration=DURATION, execute=EXECUTE_PROBE)
print("Status:", probe_report["status"], "|", PROBE_OUT)
if EXECUTE_PROBE:
    display(Audio(filename=str(PROBE_OUT / "mix_preview.mp3"), normalize=False))


In [ ]:
# Jalankan setelah probe selesai, atau setelah gagal untuk report yang tersedia.
feedback = PROBE_OUT / "full_song_feedback.zip"
if feedback.is_file():
    files.download(str(feedback))
else:
    print("Belum ada feedback inferensi. Probe dry-run tidak membuat hasil model.")


## Chunk seluruh lagu, setelah durasi konteks dipilih

Plan awal membatasi **seluruh konteks model menjadi 90 detik termasuk margin**. Jika memilih cap lain, buat plan baru dengan `context.prepare(SOURCE, NEW_OUT, context=60, margin=6, fade=.58)` dan perbarui `PLAN_PATH`. Jangan mengganti plan yang telah memiliki hasil.

Generasi sequential dan memuat model lagi per chunk. Margin/reload menambah waktu. Chunk selesai dipakai kembali setelah verifikasi; proses diffusion yang gagal diulang sebagai attempt baru. Seed sama tidak menjamin timbre/register/groove sama.


In [ ]:
EXECUTE_CHUNKS = False
if EXECUTE_CHUNKS:
    progress = context.run_chunks(ROOT, PLAN_PATH)
    report = context.stitch_run(PLAN_PATH)
    print("Sambungan perlu didengar:", [s["index"] for s in report["seams"] if s["manual_review"]])
    display(Audio(filename=str(PLAN_PATH.parent / "stitched/mix_full.mp3"), normalize=False))
else:
    print("Tidak menjalankan inferensi atau stitching; plan tersimpan di", PLAN_PATH)


In [ ]:
# Paket kecil: mix/bass MP3, report seam dan cuplikan seam; tanpa bobot model.
stitched = PLAN_PATH.parent / "stitched"
if (stitched / "seams.json").is_file():
    bundle = PLAN_PATH.parent / "chunked_feedback.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(PLAN_PATH, "plan.json")
        for path in sorted(stitched.iterdir()):
            if path.suffix in {".json", ".mp3"} or path.name.startswith("seam_"):
                archive.write(path, "stitched/" + path.name)
    files.download(str(bundle))
else:
    print("Belum ada hasil stitching lengkap.")


Equal-power crossfade **tetap dapat membatalkan fase bass**. Helper mencari kandidat berdasarkan energi/korelasi, membuang tepi model, dan melaporkan perubahan level/pitch; ia tidak menjamin sambungan musikal. Birama/downbeat tidak ditebak dari BPM. Repaint dan conditioning BPM/key belum diterapkan. Lihat `ace-context-chunks.md` di tool ZIP untuk rincian.
